In [7]:
%pip install yfinance

import yfinance as yf

ticker = yf.Ticker("qqq")

price = ticker.history(start="2009-02-14", end="2020-06-11")
print(price)


                                 Open        High         Low       Close  \
Date                                                                        
2009-02-17 00:00:00-05:00   25.455395   25.610769   25.084224   25.222334   
2009-02-18 00:00:00-05:00   25.386341   25.610770   24.920221   25.239599   
2009-02-19 00:00:00-05:00   25.394967   25.558973   24.790736   24.851160   
2009-02-20 00:00:00-05:00   24.574945   25.161913   24.505890   24.920221   
2009-02-23 00:00:00-05:00   25.058336   25.075599   23.936191   24.048407   
...                               ...         ...         ...         ...   
2020-06-04 00:00:00-04:00  228.241165  229.651185  225.681857  226.985657   
2020-06-05 00:00:00-04:00  228.134896  232.075252  227.565094  231.486130   
2020-06-08 00:00:00-04:00  231.341227  233.407981  229.767028  233.282440   
2020-06-09 00:00:00-04:00  232.422940  235.822450  232.239441  234.972580   
2020-06-10 00:00:00-04:00  236.662650  239.337845  236.141124  237.792603  

In [8]:
#Collect Features, we want them a day behind, we give yesturdays stats to guess todays price, so score follows current day
price["Change"] = price["Close"].shift(1) - price["Open"].shift(1)
price["Score"] = (price["Close"] > price["Open"]).astype(int)
price["%Change"] = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
price["YestScore"] = price["Score"].shift(1)
price["5DayMean"] = price["Close"].rolling(5).mean().shift(1)
price["5DayVoli"] = price["Close"].rolling(5).std().shift(1)
price["5DayPerf"] = price["Score"].rolling(5).sum().shift(1)

price.dropna(inplace=True)

print(price["5DayPerf"])

Date
2009-02-24 00:00:00-05:00    1.0
2009-02-25 00:00:00-05:00    2.0
2009-02-26 00:00:00-05:00    2.0
2009-02-27 00:00:00-05:00    2.0
2009-03-02 00:00:00-05:00    2.0
                            ... 
2020-06-04 00:00:00-04:00    5.0
2020-06-05 00:00:00-04:00    4.0
2020-06-08 00:00:00-04:00    4.0
2020-06-09 00:00:00-04:00    4.0
2020-06-10 00:00:00-04:00    4.0
Name: 5DayPerf, Length: 2844, dtype: float64


In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

features = ["Change", "%Change", "CloseToOpen", "YestScore", "5DayMean", "5DayVoli", "5DayPerf"]

priceTrain = price[price.index <= "2018-06-11"]
priceTest = price[price.index > "2018-06-11"]

results = []

for i in range(len(priceTest)):
    xTrain = priceTrain[features]
    yTrain = priceTrain["Score"]

    scaler = StandardScaler()
    xTrain = scaler.fit_transform(xTrain)

    model = LogisticRegression()
    model.fit(xTrain, yTrain)

    #Grab yesturdays information to predict todays, data already transformed so i is yesturday
    yest = priceTest.iloc[[i]]
    yestScal = scaler.transform(yest[features])
    pred = model.predict(yestScal)[0]
    prob = model.predict_proba(yestScal)[0][1]
    print("Analyzing Day: ", priceTest.index[i])

    results.append({"Date": priceTest.index[i],
                    "Score": priceTest["Score"][i],
                    "Prediction": pred,
                    "Probability": prob,
                    "Open": priceTest["Open"][i],
                    "Close": priceTest["Close"][i]})
    
    priceTrain = pd.concat([priceTrain, yest])

Analyzing Day:  2018-06-12 00:00:00-04:00
Analyzing Day:  2018-06-13 00:00:00-04:00
Analyzing Day:  2018-06-14 00:00:00-04:00
Analyzing Day:  2018-06-15 00:00:00-04:00
Analyzing Day:  2018-06-18 00:00:00-04:00
Analyzing Day:  2018-06-19 00:00:00-04:00
Analyzing Day:  2018-06-20 00:00:00-04:00
Analyzing Day:  2018-06-21 00:00:00-04:00
Analyzing Day:  2018-06-22 00:00:00-04:00
Analyzing Day:  2018-06-25 00:00:00-04:00
Analyzing Day:  2018-06-26 00:00:00-04:00
Analyzing Day:  2018-06-27 00:00:00-04:00
Analyzing Day:  2018-06-28 00:00:00-04:00
Analyzing Day:  2018-06-29 00:00:00-04:00
Analyzing Day:  2018-07-02 00:00:00-04:00
Analyzing Day:  2018-07-03 00:00:00-04:00
Analyzing Day:  2018-07-05 00:00:00-04:00
Analyzing Day:  2018-07-06 00:00:00-04:00
Analyzing Day:  2018-07-09 00:00:00-04:00
Analyzing Day:  2018-07-10 00:00:00-04:00
Analyzing Day:  2018-07-11 00:00:00-04:00
Analyzing Day:  2018-07-12 00:00:00-04:00
Analyzing Day:  2018-07-13 00:00:00-04:00
Analyzing Day:  2018-07-16 00:00:0

In [10]:
results = pd.DataFrame(results)

score = results["Score"]
pred = results["Prediction"]
op = results["Open"]
close = results["Close"]

accuracy = accuracy_score(score, pred)
precision = precision_score(score, pred, zero_division=0)
recall = recall_score(score, pred, zero_division=0)
f1 = f1_score(score, pred, zero_division=0)

print("Accuracy: ", accuracy, "\nPrecision: ", precision, "\nRecall: ", recall, "\nF1: ", f1)



Accuracy:  0.5228628230616302 
Precision:  0.5348837209302325 
Recall:  0.7752808988764045 
F1:  0.6330275229357798


In [11]:
#Find out how much you would have made/lost with this method
totalIn = 0
totalOut = 0
totalInvested = 0
totalStoodOut = 0
alltradein = 0
alltradeout = 0
perftradein = 0
perftradeout = 0
for i, o, c, s in zip(pred.values, op.values, close.values, score.values):
    if i == 1:
        totalIn += 100
        totalInvested += 1
        temp = ((c - o)/o) * 100
        totalOut += 100 + temp
        print(temp, o, c)
    else:
        totalStoodOut += 1
    alltradein += 100
    alltradeout += (((c-o)/o) * 100) + 100
    if s == 1:
        perftradein += 100
        perftradeout += (((c-o)/o) * 100) + 100

print(totalIn)
print(totalOut)
print(totalInvested)
print(totalStoodOut)
print(alltradein)
print(alltradeout)
print(perftradein)
print(perftradeout)

0.3882431962699286 166.28236801762193 166.92794799804688
-0.15899986916087894 167.1843256666847 166.9185028076172
0.5491712584997661 167.68752202125785 168.60841369628906
0.028261489757387313 167.97227340620293 168.01974487304688
0.5755282211073044 166.96178677326424 167.92269897460938
0.9927099817099423 165.80106722269556 167.44699096679688
0.24885608703231307 168.21761070505016 168.63623046875
-1.1198594645753466 169.064411128588 167.17112731933594
-0.5558682419598938 167.7324354269182 166.80006408691406
-1.3584301320382 165.2872892705918 163.04197692871094
0.040701932685317514 163.64135684122596 163.7079620361328
-1.7766245775465674 164.4025047587715 161.481689453125
0.9851333924350321 161.28188582584318 162.8707275390625
1.623153301929759 161.77659887274888 164.40248107910156
0.5875135159796028 163.5557807968867 164.51669311523438
1.3212515516740597 164.8972507675194 167.07595825195312
-0.1126616400567655 168.89312810986863 168.70285034179688
0.2785229998088472 167.38038002045215 1